In [25]:
import pandas as pd
import glob

# Load only lightweight columns to avoid memory crash
COLS = [
    "conversation_hash", "model", "timestamp", "turn",
    "language", "toxic", "redacted", "state", "country", "hashed_ip"
]

files = sorted(glob.glob("../Data/WildChatData/*.parquet"))
df = pd.concat([pd.read_parquet(f, columns=COLS) for f in files], ignore_index=True)

In [26]:
print("Shape:", df.shape)
print("\nColumn types:")
display(df.dtypes.to_frame("dtype"))
print("\nFirst 5 rows:")
display(df.head())
print("\nDataset info:")
df.info()
print("\nNumeric summary:")
display(df.describe())

Shape: (2523967, 10)

Column types:


,dtype
conversation_hash,str
model,str
timestamp,"datetime64[us, UTC]"
turn,int64
language,str
toxic,bool
redacted,bool
state,str
country,str
hashed_ip,str



First 5 rows:


,conversation_hash,model,timestamp,turn,language,toxic,redacted,state,country,hashed_ip
0,c9ec5b440fbdd2a269333dd241f32f64,gpt-4-0314,2023-04-09 00:02:53+00:00,1,English,False,False,Texas,United States,22fd87ba9b98f3d379b23c7b52961f2d4a8505127e58b3...
1,34f1581760df304d539e2fe4653b40d3,gpt-4-0314,2023-04-09 00:03:20+00:00,2,Spanish,False,False,A Coruña,Spain,58369722cd0bdf7fc027a67491ba65b74576df6994c36c...
2,cf1267ca6b2f6fccc9c36652a00059a1,gpt-4-0314,2023-04-09 00:04:52+00:00,1,English,False,False,Mecca Region,Saudi Arabia,8133108d1c433c180c6be8302dc5a6681f2bec980190a1...
3,7f1c97a4f873cda8106b010d040be078,gpt-4-0314,2023-04-09 00:06:29+00:00,1,Catalan,False,False,Barcelona,Spain,846e43fb5fbb4b8cfbafa17083387aad62e58f5fb23482...
4,e98d3e74c57f9a65261df393d9124ac2,gpt-4-0314,2023-04-09 00:06:49+00:00,1,English,False,False,Texas,United States,22fd87ba9b98f3d379b23c7b52961f2d4a8505127e58b3...



Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 2523967 entries, 0 to 2523966
Data columns (total 10 columns):
 #   Column             Dtype              
---  ------             -----              
 0   conversation_hash  str                
 1   model              str                
 2   timestamp          datetime64[us, UTC]
 3   turn               int64              
 4   language           str                
 5   toxic              bool               
 6   redacted           bool               
 7   state              str                
 8   country            str                
 9   hashed_ip          str                
dtypes: bool(2), datetime64[us, UTC](1), int64(1), str(6)
memory usage: 489.7 MB

Numeric summary:


,turn
count,2.523967e+06
mean,2.338418e+00
std,3.014108e+00
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,2.000000e+00
max,2.490000e+02


In [27]:
df.columns.tolist()

['conversation_hash',
 'model',
 'timestamp',
 'turn',
 'language',
 'toxic',
 'redacted',
 'state',
 'country',
 'hashed_ip']

In [28]:
# Frequency tables for categorical columns
print("=== Model usage ===")
print(df["model"].value_counts(), "\n")

print("=== Top languages ===")
print(df["language"].value_counts().head(20), "\n")

print("=== Top countries ===")
print(df["country"].value_counts().head(20), "\n")

print("=== Toxic flag ===")
print(df["toxic"].value_counts(), "\n")

print("=== Redacted flag ===")
print(df["redacted"].value_counts(), "\n")

print("=== Missing values per column ===")
print(df.isnull().sum())

=== Model usage ===
model
gpt-3.5-turbo-0613    1118256
gpt-3.5-turbo-0301     587550
gpt-4-1106-preview     297917
gpt-4-0125-preview     186619
gpt-3.5-turbo-0125     173126
gpt-4-0314             160493
gpt-4-0613                  6
Name: count, dtype: int64 

=== Top languages ===
language
English       1443384
Chinese        358597
Russian        261902
French          81091
Spanish         61475
German          50683
Arabic          35838
Portuguese      31229
Turkish         18353
Italian         15380
Vietnamese      14271
Persian         11750
Nolang          11155
Polish          11016
Japanese        10499
Maori            9861
Latin            9713
Korean           9649
Indonesian       9210
Sotho            6583
Name: count, dtype: int64 

=== Top countries ===
country
United States      514151
Russia             342704
China              304444
Hong Kong          141400
United Kingdom      92918
Germany             87194
France              78517
Japan               52901

In [29]:
# Save combined data as a single parquet file (faster + smaller than CSV)
df.to_parquet("../Data/WildChatData/wildchat_combined.parquet", index=False)

In [30]:
# Load from combined file (use this instead of cell 1 once the file exists)
df = pd.read_parquet("../Data/WildChatData/wildchat_combined.parquet")

In [31]:
# Filter to English conversations only, then take a random sample of 5000 rows
english_df = df[df["language"] == "English"]
sample = english_df.sample(n=5000, random_state=42)

# Save the sample as parquet and CSV
sample.to_parquet("../Data/WildChatData/wildchat_sample_5000.parquet", index=False)
sample.to_csv("../Data/WildChatData/wildchat_sample_5000.csv", index=False)

print(f"Full dataset:         {df.shape[0]:,} rows")
print(f"English only:         {english_df.shape[0]:,} rows")
print(f"Sample size:          {sample.shape[0]:,} rows")
print("Saved to wildchat_sample_5000.parquet and wildchat_sample_5000.csv")
display(sample.head())

Full dataset:         2,523,967 rows
English only:         1,443,384 rows
Sample size:          5,000 rows
Saved to wildchat_sample_5000.parquet and wildchat_sample_5000.csv


,conversation_hash,model,timestamp,turn,language,toxic,redacted,state,country,hashed_ip
796554,a022c8720f12ac973d03ae39c6fc4bfb,gpt-4-0125-preview,2024-04-11 02:03:01+00:00,7,English,False,False,Queensland,Australia,349c9db594c1e9427204606f5e2fa618ea5bab7b5db4b5...
2311429,97f318ba0196f7324d87598684fc10c5,gpt-3.5-turbo-0613,2023-12-28 11:33:35+00:00,1,English,False,False,New Taipei,Taiwan,764f1c351dc41c76e20c42a8cffb65b0f0ffe3b0a0ab65...
988632,13f4fa32225c7ce4ab532ea705418ff1,gpt-3.5-turbo-0301,2023-05-30 13:08:30+00:00,1,English,False,False,Shanghai,China,ec20d566b4ef63fb1a31f4ef8e7e298c5c01b7089f9bb9...
1745679,04e18920923d5cf05c249782c8f5b415,gpt-3.5-turbo-0301,2023-05-07 06:31:18+00:00,1,English,False,False,Limburg Province,Belgium,dadc4bfe9a7133bf0a3068a6c6285064df6d5111a12dfb...
1079903,66d6f125f9f370c86e4e8ee4abb6dfd1,gpt-3.5-turbo-0301,2023-06-24 00:39:31+00:00,2,English,False,False,North Carolina,United States,30549061e0cde1ad08384cd5fff860db908d50d47ac633...


In [33]:
# Re-attach conversations to the current English sample
sample_hashes = set(sample["conversation_hash"])

# Use the original numbered source files only — generated files don't have conversation column
source_files = sorted(glob.glob("../Data/WildChatData/[0-9][0-9][0-9][0-9].parquet"))

conv_frames = []
for f in source_files:
    chunk = pd.read_parquet(f)[["conversation_hash", "conversation"]]
    conv_frames.append(chunk[chunk["conversation_hash"].isin(sample_hashes)])

conv_df = pd.concat(conv_frames, ignore_index=True)
sample_with_conv = sample.merge(conv_df, on="conversation_hash", how="left")

# Save as parquet (preserves nested conversation structure)
sample_with_conv.to_parquet("../Data/WildChatData/wildchat_sample_5000_with_conv.parquet", index=False)

# Save as CSV — conversations will be stored as text representations of the list
sample_with_conv.to_csv("../Data/WildChatData/wildchat_sample_5000_with_conv.csv", index=False)

print(f"Rows saved: {sample_with_conv.shape[0]:,}")
print(f"Columns:    {sample_with_conv.columns.tolist()}")
print("Saved to wildchat_sample_5000_with_conv.parquet and wildchat_sample_5000_with_conv.csv")

Rows saved: 9,941
Columns:    ['conversation_hash', 'model', 'timestamp', 'turn', 'language', 'toxic', 'redacted', 'state', 'country', 'hashed_ip', 'conversation']
Saved to wildchat_sample_5000_with_conv.parquet and wildchat_sample_5000_with_conv.csv


In [34]:
non_english = df[df["language"] != "English"]

print(f"Total rows:       {len(df):,}")
print(f"English:          {len(english_df):,} ({len(english_df)/len(df)*100:.1f}%)")
print(f"Non-English:      {len(non_english):,} ({len(non_english)/len(df)*100:.1f}%)")
print()
print("Top non-English languages:")
display(non_english["language"].value_counts().head(15).to_frame("count"))

Total rows:       2,523,967
English:          1,443,384 (57.2%)
Non-English:      1,080,583 (42.8%)

Top non-English languages:


,count
language,
Chinese,358597
Russian,261902
French,81091
Spanish,61475
German,50683
Arabic,35838
Portuguese,31229
Turkish,18353
Italian,15380
